# GuardianAire Thermal Person Detection
Fine-tune YOLO11-N on UAV thermal imagery for person detection.

**Runtime:** Go to Runtime → Change runtime type → **T4 GPU**

**Time estimate:** ~45 min for 100 epochs on T4

## 1. Setup

In [ ]:
!pip install -q ultralytics

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Download and Prepare HIT-UAV Dataset
High-altitude infrared thermal dataset from UAV perspective.
Dataset is in VOC XML format — we convert to YOLO format.

In [ ]:
from pathlib import Path

DATASET_DIR = Path("/content/datasets/hit-uav")

if not DATASET_DIR.exists():
    print("Downloading HIT-UAV dataset...")
    !git clone --depth 1 https://github.com/suojiashun/HIT-UAV-Infrared-Thermal-Dataset.git {DATASET_DIR}
    print("Done!")
else:
    print(f"Dataset already exists at {DATASET_DIR}")

# Show structure
!find {DATASET_DIR} -type d -not -path '*/.git/*' | head -10

In [ ]:
import xml.etree.ElementTree as ET
import shutil
import random

SRC_DIR = Path("/content/datasets/hit-uav/normal_xml")
OUT_DIR = Path("/content/datasets/hit-uav-yolo")

CLASS_MAP = {"Person": 0, "Car": 1, "Bicycle": 2, "OtherVehicle": 1}

for split in ["train", "val", "test"]:
    (OUT_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

# Get train/val/test splits from ImageSets
splits = {}
for split_name in ["train", "val", "test"]:
    split_file = SRC_DIR / "ImageSets" / "Main" / f"{split_name}.txt"
    if split_file.exists():
        splits[split_name] = [l.strip() for l in split_file.read_text().splitlines() if l.strip()]
        print(f"{split_name}: {len(splits[split_name])} images")

# If no splits, create 70/15/15
if not splits:
    all_imgs = sorted((SRC_DIR / "JPEGImages").glob("*"))
    random.seed(42)
    random.shuffle(all_imgs)
    n = len(all_imgs)
    splits = {
        "train": [p.stem for p in all_imgs[:int(n * 0.7)]],
        "val": [p.stem for p in all_imgs[int(n * 0.7):int(n * 0.85)]],
        "test": [p.stem for p in all_imgs[int(n * 0.85):]],
    }
    for s, ids in splits.items():
        print(f"{s}: {len(ids)} images")

# Convert VOC XML to YOLO format
total = 0
for split_name, img_ids in splits.items():
    for img_id in img_ids:
        img_file = None
        for ext in [".jpg", ".jpeg", ".png", ".bmp"]:
            candidate = SRC_DIR / "JPEGImages" / f"{img_id}{ext}"
            if candidate.exists():
                img_file = candidate
                break
        if not img_file:
            continue

        shutil.copy(img_file, OUT_DIR / "images" / split_name / img_file.name)

        xml_file = SRC_DIR / "Annotations" / f"{img_id}.xml"
        if not xml_file.exists():
            (OUT_DIR / "labels" / split_name / f"{img_id}.txt").write_text("")
            total += 1
            continue

        tree = ET.parse(xml_file)
        root = tree.getroot()
        size = root.find("size")
        img_w = int(size.find("width").text)
        img_h = int(size.find("height").text)

        lines = []
        for obj in root.findall("object"):
            cls_name = obj.find("name").text
            if cls_name not in CLASS_MAP:
                continue
            cls_id = CLASS_MAP[cls_name]
            bbox = obj.find("bndbox")
            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)
            cx = ((xmin + xmax) / 2) / img_w
            cy = ((ymin + ymax) / 2) / img_h
            bw = (xmax - xmin) / img_w
            bh = (ymax - ymin) / img_h
            lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

        (OUT_DIR / "labels" / split_name / f"{img_id}.txt").write_text("\n".join(lines))
        total += 1

print(f"\nConverted {total} images to YOLO format at {OUT_DIR}")

In [ ]:
# Write dataset YAML
yaml_content = """path: /content/datasets/hit-uav-yolo
train: images/train
val: images/val
test: images/test

names:
  0: Person
  1: Car
  2: Bicycle
"""

with open('/content/hit_uav.yaml', 'w') as f:
    f.write(yaml_content)

print("Dataset config written to /content/hit_uav.yaml")
print()

# Verify
for split in ['train', 'val', 'test']:
    n_imgs = len(list((OUT_DIR / 'images' / split).glob('*')))
    n_lbls = len(list((OUT_DIR / 'labels' / split).glob('*')))
    print(f"  {split}: {n_imgs} images, {n_lbls} labels")

## 3. Preview Sample Images

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np

train_imgs = sorted((OUT_DIR / 'images' / 'train').glob('*'))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flat, train_imgs):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')

plt.suptitle('HIT-UAV Thermal Training Samples', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Train YOLO11-N
Fine-tune the nano model with thermal-optimized augmentation.
~45 minutes on T4 GPU for 100 epochs.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

results = model.train(
    data='/content/hit_uav.yaml',
    epochs=100,
    imgsz=640,
    batch=-1,
    device=0,
    project='/content/runs/thermal',
    name='train_yolo11n',
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.3,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    flipud=0.1,
    mosaic=0.8,
    mixup=0.1,
    patience=20,
    save=True,
    plots=True,
)

## 5. Evaluate Results

In [ ]:
best_model = YOLO('/content/runs/thermal/train_yolo11n/weights/best.pt')

metrics = best_model.val(
    data='/content/hit_uav.yaml',
    split='test',
    plots=True,
)

print(f"\n=== Test Results ===")
print(f"mAP50:     {metrics.box.map50:.3f}")
print(f"mAP50-95:  {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")

In [ ]:
# Visualize predictions on test images
test_imgs = sorted((OUT_DIR / 'images' / 'test').glob('*'))[:8]

preds = best_model.predict(
    source=[str(p) for p in test_imgs],
    imgsz=640,
    conf=0.25,
    save=True,
    project='/content/runs/thermal',
    name='test_predictions',
)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
pred_dir = Path('/content/runs/thermal/test_predictions')
pred_imgs = sorted(pred_dir.glob('*'))[:8]

for ax, img_path in zip(axes.flat, pred_imgs):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')

plt.suptitle('Thermal Person Detection - Test Predictions', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Training Curves

In [ ]:
from IPython.display import Image, display

results_img = Path('/content/runs/thermal/train_yolo11n/results.png')
if results_img.exists():
    display(Image(filename=str(results_img), width=900))

cm_img = Path('/content/runs/thermal/train_yolo11n/confusion_matrix.png')
if cm_img.exists():
    display(Image(filename=str(cm_img), width=600))

## 7. Export to TFLite for Android

In [ ]:
tflite_path = best_model.export(
    format='tflite',
    imgsz=640,
    half=True,
)

tflite_file = Path(tflite_path)
size_mb = tflite_file.stat().st_size / (1024 * 1024)
print(f"\nTFLite FP16 model: {tflite_path}")
print(f"Size: {size_mb:.1f} MB")

tflite_int8_path = best_model.export(
    format='tflite',
    imgsz=640,
    int8=True,
)

tflite_int8_file = Path(tflite_int8_path)
size_int8_mb = tflite_int8_file.stat().st_size / (1024 * 1024)
print(f"\nTFLite INT8 model: {tflite_int8_path}")
print(f"Size: {size_int8_mb:.1f} MB")

## 8. Download Models

In [ ]:
from google.colab import files
import shutil

export_dir = Path('/content/export')
export_dir.mkdir(exist_ok=True)

shutil.copy('/content/runs/thermal/train_yolo11n/weights/best.pt', export_dir / 'thermal_person_best.pt')
if tflite_file.exists():
    shutil.copy(tflite_file, export_dir / 'thermal_person_fp16.tflite')
if tflite_int8_file.exists():
    shutil.copy(tflite_int8_file, export_dir / 'thermal_person_int8.tflite')

shutil.make_archive('/content/thermal_person_models', 'zip', export_dir)
print("Models packaged!")

files.download('/content/thermal_person_models.zip')

## 9. Quick Benchmark

In [ ]:
import time

test_img = str(test_imgs[0])

for _ in range(5):
    best_model.predict(test_img, imgsz=640, verbose=False)

times = []
for _ in range(50):
    t0 = time.perf_counter()
    best_model.predict(test_img, imgsz=640, verbose=False)
    times.append((time.perf_counter() - t0) * 1000)

avg_ms = sum(times) / len(times)
print(f"GPU inference: {avg_ms:.1f} ms/frame ({1000/avg_ms:.0f} FPS)")
print(f"\nEstimated Android: {avg_ms * 5:.0f}-{avg_ms * 10:.0f} ms/frame ({1000/(avg_ms*10):.0f}-{1000/(avg_ms*5):.0f} FPS)")

---
## Next Steps

1. **Download** the exported models (cell above)
2. **Copy TFLite** to `autel-mobilesdk-2.0/app/src/main/assets/`
3. **Integrate** TFLite inference into the Android app
4. **Collect** your own thermal data from drone flights for fine-tuning
5. **Iterate** — retrain with your custom data + HIT-UAV combined